# Data Processing

In [ ]:
import os
import sys
import torch
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from facial_emotion_recognition import EmotionRecognition
import mediapipe as mp
from tqdm import tqdm
import logging
import pympi
import gc

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
device = "cuda" if torch.cuda.is_available() else "cpu"
logging.info(f"Using device: {device}")

2025-11-26 16:32:41,444 [INFO] Using device: cuda


In [16]:
def create_labels_from_filenames(root_dir):
    labels_dict = {}
    skipped_files = []
    
    print(f"📂 Scanning {root_dir} for labels...")
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if not file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                continue
            
            name_lower = file.lower()
            
            if 'lie' in name_lower:
                labels_dict[file] = 1
            elif 'truth' in name_lower:
                labels_dict[file] = 0
            else:
                skipped_files.append(file)

    print(f"✅ Found {len(labels_dict)} labeled videos.")
    if skipped_files:
        print(f"⚠️ Warning: Could not determine label for {len(skipped_files)} videos (e.g., {skipped_files[:3]}).")
        
    return labels_dict

# Segmenting and .eaf parsing

In [3]:
import abc

class BaseSegmenter(abc.ABC):
    @abc.abstractmethod
    def get_segments(self, video_path):
        pass

class SilesianSegmenter(BaseSegmenter):
    def __init__(self, fps=100):
        self.fps = fps

    def _convert_timestamp(self, timestamp_ms):
        return int((timestamp_ms / 1000.0) * self.fps)

    def get_segments(self, video_path):
        eaf_path = video_path.replace('.avi', '.eaf')
        if not os.path.exists(eaf_path):
            logging.warning(f"Annotation file missing: {eaf_path}")
            return []
        try:
            eaf = pympi.Elan.Eaf(eaf_path)
            annotations = eaf.get_annotation_data_for_tier('Question')
        except Exception as e:
            logging.error(f"Failed to parse EAF {eaf_path}: {e}")
            return []
        
        segments = []
        for i, (start, end, value) in enumerate(annotations):
            if value == 'Correct':
                is_deceptive = 1 if (i not in [0, 1, 8]) else 0 
                
                segments.append((
                    self._convert_timestamp(start), 
                    self._convert_timestamp(end), 
                    is_deceptive
                ))
        return segments

class SimpleLabelSegmenter(BaseSegmenter):
    """
    For datasets where 1 video = 1 label.
    Expects a dictionary mapping filenames to labels.
    """
    def __init__(self, label_map, video_fps=30):
        self.label_map = label_map
        self.fps = video_fps

    def get_segments(self, video_path):
        filename = os.path.basename(video_path)
        if filename not in self.label_map:
            return []
        
        label = self.label_map[filename]
        
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        
        return [(0, total_frames, label)]

### Face detection and crop (YOLO)

In [4]:
def detect_faces(model, frame):
    results = model(frame, verbose=False)
    if not results or results[0].boxes is None:
        return []
    return results[0].boxes.xyxy.int().tolist()

def face_crop(model, frame):
    boxes = detect_faces(model, frame)

    for _, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        face_crop = frame[y1:y2, x1:x2]
        if face_crop.size == 0:
            continue
        return face_crop
    
    return None

### Resize images to consistent size

In [5]:
def resize_frame(frame, size=(224, 224)):
    return cv2.resize(frame, size)

### Geometric face normalization with MediaPipe

In [6]:
def geometric_normalization(frame, face_mesh):
    LEFT_EYE_LANDMARKS = [33, 133]
    RIGHT_EYE_LANDMARKS = [362, 263]

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_frame)

    if not results.multi_face_landmarks:
        return frame

    landmarks = results.multi_face_landmarks[0].landmark
    h, w, _ = frame.shape

    left_eye = np.array([[landmarks[i].x * w, landmarks[i].y * h] for i in LEFT_EYE_LANDMARKS]).mean(axis=0)
    right_eye = np.array([[landmarks[i].x * w, landmarks[i].y * h] for i in RIGHT_EYE_LANDMARKS]).mean(axis=0)

    dy = right_eye[1] - left_eye[1]
    dx = right_eye[0] - left_eye[0]
    angle = np.degrees(np.arctan2(dy, dx))

    center = tuple(map(float, np.mean([left_eye, right_eye], axis=0)))
    rot_mat = cv2.getRotationMatrix2D(center, angle, 1.0)
    aligned = cv2.warpAffine(frame, rot_mat, (w, h), flags=cv2.INTER_CUBIC)

    return aligned

### Emotion Detection

In [7]:
def get_emotion_probs(frame, emotion_detector):
    if frame.ndim == 3:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    tensor = emotion_detector.transform(frame).unsqueeze(0).to(emotion_detector.device)

    with torch.no_grad():
        output = emotion_detector.network(tensor)
        probs = torch.softmax(output, dim=1).cpu().numpy()[0]

    return {emotion_detector.emotions[i]: float(probs[i]) for i in range(len(probs))}

def detect_emotions(frame, emotion_detector):
    return get_emotion_probs(frame, emotion_detector)

### Face Landmarks

In [8]:
def extract_landmarks(frame, face_mesh):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    if not results.multi_face_landmarks:
        return None
    pts = results.multi_face_landmarks[0].landmark
    return np.array([(p.x, p.y) for p in pts], dtype=np.float32).flatten()

### Optical Flow

In [9]:
def compute_optical_flow(prev_gray, gray):
    flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None,
                                        pyr_scale=0.5, levels=3, winsize=15,
                                        iterations=3, poly_n=5, poly_sigma=1.1, flags=0)
    return {
        "flow_mean_x": float(flow[...,0].mean()),
        "flow_mean_y": float(flow[...,1].mean()),
        "flow_std_x": float(flow[...,0].std()),
        "flow_std_y": float(flow[...,1].std())
    }

### All together

In [10]:
def process_segment(video_cap, start_frame, end_frame, label, sample_id, face_detector, emotion_detector, face_mesh, frame_skip):
    results = []
    
    video_cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    current_frame = start_frame
    processed_count = 0
    prev_gray = None

    while current_frame <= end_frame:
        ret, frame = video_cap.read()
        if not ret:
            break

        if processed_count % frame_skip != 0:
            current_frame += 1
            processed_count += 1
            continue

        face = face_crop(face_detector, frame)
        if face is None:
            current_frame += 1
            processed_count += 1
            continue

        landmarks = extract_landmarks(face, face_mesh)
        if landmarks is None:
            landmarks = [0.0] * (478*2)

        resized_face = resize_frame(face)
        gray = cv2.cvtColor(resized_face, cv2.COLOR_BGR2GRAY)
        
        if prev_gray is not None:
            flow = compute_optical_flow(prev_gray, gray)
        else:
            flow = {"flow_mean_x": 0.0, "flow_mean_y": 0.0, "flow_std_x": 0.0, "flow_std_y": 0.0}
        prev_gray = gray

        normalized_face = geometric_normalization(resized_face, face_mesh)
        emotions = detect_emotions(normalized_face, emotion_detector)

        results.append({
            'id': sample_id,
            'frame': current_frame,
            'deceptive': label,
            **{f"lm_{i}": landmarks[i] for i in range(len(landmarks))},
            **emotions,
            **flow
        })

        current_frame += 1
        processed_count += 1

    return results

In [11]:
def process_video(sample_id, video_path, segmenter, 
                            face_detector, emotion_detector, face_mesh, frame_skip):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        logging.error(f"Could not open {video_path}")
        return sample_id, []

    segments = segmenter.get_segments(video_path)
    
    if not segments:
        cap.release()
        return sample_id, []

    logging.info(f"Processing {video_path}: Found {len(segments)} segments.")
    
    all_video_results = []
    
    for start, end, label in segments:
        segment_results = process_segment(
            cap, start, end, label, sample_id,
            face_detector, emotion_detector, face_mesh, frame_skip
        )
        
        if len(segment_results) > 0:
            all_video_results.extend(segment_results)
            sample_id += 1 

    cap.release()
    return sample_id, all_video_results

In [12]:
def process_dataset(
    root_dir, 
    out_path, 
    dataset_type='silesian', # 'silesian' or 'simple'
    labels_dict=None,        # Needed if type='simple'
    frame_skip=5, 
    device=device
):
    logging.info(f"Starting processing for {dataset_type} dataset...")

    face_detector = YOLO('model_weights/yolov8n-face.pt').to(device)
    emotion_detector = EmotionRecognition(device='gpu' if device == 'cuda' else 'cpu')
    
    if dataset_type == 'silesian':
        segmenter = SilesianSegmenter()
    elif dataset_type == 'simple':
        if labels_dict is None:
            raise ValueError("labels_dict is required for 'simple' dataset type")
        segmenter = SimpleLabelSegmenter(labels_dict)
    else:
        raise ValueError(f"Unknown dataset type: {dataset_type}")

    sample_id = 0
    
    header_written = False
    if os.path.exists(out_path):
        os.remove(out_path)

    mp_face_mesh = mp.solutions.face_mesh
    with mp_face_mesh.FaceMesh(
        static_image_mode=True,
        refine_landmarks=True,
        max_num_faces=1
    ) as face_mesh:
        
        video_files = []
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.lower().endswith(('.avi', '.mp4', '.mov')):
                    video_files.append(os.path.join(root, file))

        for video_path in tqdm(video_files, desc="Processing Videos"):
            
            sample_id, results = process_video(
                sample_id, video_path, segmenter,
                face_detector, emotion_detector, face_mesh, frame_skip
            )
            
            if len(results) > 0:
                df = pd.DataFrame(results)
                df.to_csv(out_path, mode="a", index=False, header=not header_written)
                header_written = True

            gc.collect()
            torch.cuda.empty_cache()

    logging.info("Dataset processing complete!")

# Real Life Deception Detection

In [ ]:
labels_dict = create_labels_from_filenames('../data/real_life_deception_detection_dataset')
process_dataset(root_dir='../data/real_life_deception_detection_dataset', out_path='../processed_data/real_life_deception_detection_dataset/emotions_landmarks_flow.csv', dataset_type='simple', labels_dict=labels_dict, frame_skip=5, device=device)

📂 Scanning data/real_life_deception_detection_dataset for labels...
✅ Found 121 labeled videos.
2025-11-26 16:40:48,490 [INFO] Starting processing for simple dataset...


I0000 00:00:1764171648.583879   10170 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1764171648.615513   10403 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.95.05), renderer: NVIDIA GeForce RTX 2060/PCIe/SSE2


[*] Accuracy: 0.9565809379727686


W0000 00:00:1764171648.617021   10399 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Processing Videos:   0%|          | 0/121 [00:00<?, ?it/s]

2025-11-26 16:40:48,625 [INFO] Processing data/real_life_deception_detection_dataset/Test/trial_lie_057.mp4: Found 1 segments.


...

2025-11-26 16:51:19,398 [INFO] Processing data/real_life_deception_detection_dataset/Train/trial_truth_053.mp4: Found 1 segments.


Processing Videos:  99%|█████████▉| 120/121 [10:36<00:03,  3.67s/it]

2025-11-26 16:51:24,984 [INFO] Processing data/real_life_deception_detection_dataset/Train/trial_truth_054.mp4: Found 1 segments.


Processing Videos: 100%|██████████| 121/121 [10:41<00:00,  5.30s/it]

2025-11-26 16:51:29,987 [INFO] Dataset processing complete!


## Silesian Deception Dataset

In [ ]:
process_dataset(root_dir='../data/silesian_deception_dataset', out_path='../processed_data/silesian_deception_dataset/emotions_landmarks_flow.csv', dataset_type='silesian', frame_skip=5, device=device)

2025-11-15 18:17:48,907 [INFO] 🚀 Starting dataset processing...


I0000 00:00:1763227069.252159    4002 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1763227069.295453    4062 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.95.05), renderer: NVIDIA GeForce RTX 2060/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


[*] Accuracy: 0.9565809379727686
Processing folders:   0%|                                  | 0/3 [00:00<?, ?it/s]

W0000 00:00:1763227069.297926    4057 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


2025-11-15 18:17:49,308 [INFO] Processing video: data/silesian_deception_dataset/poli2Video/person1.avi


W0000 00:00:1763227069.306520    4058 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/pekoraptor/dev/lie-detection/.venv/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


2025-11-15 18:19:02,054 [INFO] ✅ Finished video: data/silesian_deception_dataset/poli2Video/person1.avi (1961 frames processed)
2025-11-15 18:19:03,788 [INFO] Processing video: data/silesian_deception_dataset/poli2Video/person10.avi
2025-11-15 18:20:05,608 [INFO] ✅ Finished video: data/silesian_deception_dataset/poli2Video/person10.avi (1697 frames processed)
2025-11-15 18:20:07,129 [INFO] Processing video: data/silesian_deception_dataset/poli2Video/person11.avi
2025-11-15 18:21:25,470 [INFO] ✅ Finished video: data/silesian_deception_dataset/poli2Video/person11.avi (2156 frames processed)
2025-11-15 18:21:27,391 [INFO] Processing video: data/silesian_deception_dataset/poli2Video/person12.avi
2025-11-15 18:22:42,286 [INFO] ✅ Finished video: data/silesian_deception_dataset/poli2Video/person12.avi (1975 frames processed)
2025-11-15 18:22:44,053 [INFO] Processing video: data/silesian_deception_dataset/poli2Video/person13.avi
2025-11-15 18:23:55,849 [INFO] ✅ Finished video: data/silesian_de